# Week 5 - Bivariate Analysis
Please run the cells of the notebook as you get to them while reading.

In [ ]:
import os
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import seaborn as sns
import matplotlib.pyplot as plt
from IPython.display import Image, display

# 1. Lesson: Bivariate graphs
In this lesson, we'll consider some bivariate plots: scatter plots, line plots, waterfall plots, area plots, and pair plots. We'll also look at bivariate statistics such as correlations, ratios, and percentages. First, let's create some data.

In [ ]:
np.random.seed(0)
numpoints = 100
lesson_series_uniform = np.round(np.random.random(size = numpoints) * 100, 2)
lesson_series_base = np.round(np.random.random(size = numpoints)**2 * 100, 2)
lesson_series_skewed = lesson_series_base + np.random.normal(scale = 10, size = numpoints)
lesson_series_reverse = 100 - lesson_series_base + np.random.normal(scale = 10, size = numpoints)

This scatter plot puts one series against another. The two series are anticorrelated - one goes up when the other goes down.

In [ ]:
sns.scatterplot(x = lesson_series_skewed, y = lesson_series_reverse)

This pair plot compares three series against one another. The diagonal shows the histogram of each series, while the non-diagonal entries plot two different series against each other. Because series #1 is not based on the "base" series, it is not correlated with series #2 or series #3. The two scatter plots that compare series #2 with series #3 do appear to be correlated due to the "base" series that they share in common.

In [ ]:
df = pd.DataFrame(np.array([lesson_series_uniform, lesson_series_skewed, lesson_series_reverse]).T, columns = ['uniform', 'skewed', 'reverse'])

In [ ]:
sns.pairplot(df)

We can compute the correlation coefficient between each pair.

In [ ]:
print("Correlations:", df.corr())

This shows a strong negative correlation between the "skewed" and "reverse" datasets and almost no correlation between the "uniform" dataset and the others.

**What is a correlation telling us?** R near 1: both variables tend to be large or small together (points lie near a rising line). R near -1: when one is large the other is small (falling line). R near 0: no linear relationship (a cloud). R² is always positive and smaller in magnitude than R unless R is 0 or ±1.

**Uses of correlation:** finding useful predictors, spotting redundant variables (e.g. height in inches vs centimetres), detecting data problems, and reporting interesting relationships.

A waterfall chart shows a value after a sequence of inputs and outputs.

In [ ]:
# Requires plotly: run `pip install plotly` if this cell fails
try:
    import plotly.graph_objects as go

    fig = go.Figure(go.Waterfall(
        name='This title is not displayed',
        orientation='v',
        measure=['absolute','relative','relative','relative','relative'],
        x=["2023-Jan", "receipts-in", "interest-in", "expenses-out", "interest-out", "2024-Jan"],
        textposition='outside',
        text=['Total','after-receipts','after-interest','after-expenses','after-interest'],
        y=[123.45,42.08,21.97,-54.31,-50.23],
        connector=dict(line=dict(color='rgb(63, 63, 63)'))
    ))
    fig.update_layout(title='Waterfall Plot', showlegend=False)
    fig.update_yaxes(range=[-1,230])
    fig.show()
except ImportError:
    print("plotly is not installed - run: pip install plotly")

A line plot is typically associated with a time or date range. It shows how a variable rises and falls over a period of time.

In [ ]:
numdates = 12
time_series = np.random.random(size = numdates) * 5 + np.arange(numdates) / 3
# 'ME' = month end (pandas 2.2+ replaced the old 'M' alias)
datearray = pd.date_range(start='2023/01/01', end='2024/01/01', freq='ME')
sns.lineplot(x = datearray, y = time_series)

An area plot stacks one curve on top of another. The top line shows ser1 + ser2 while the bottom line shows ser1.

In [ ]:
time_series_2 = np.random.random(size = numdates) * 5 + np.arange(numdates) / 3
df = pd.DataFrame({'ser1': time_series, 'ser2': time_series_2}, index = datearray)
df.plot.area()

# 2. Weekly graph question
Below is a pair plot showing the correlation between the levels of chemical A, B, and C in a water sample. If you had to choose just three of the nine graphs in the pair plot to display, which would you choose? How would you alter the way these graphs are displayed to make them clearer or more informative? Would you fit any of these graphs to a line or curve? Why or why not? What if you had to choose just one? What if you had to choose six?

In [ ]:
np.random.seed(0)
num_data = 100
base_data = np.random.random(size = num_data)
chem1 = base_data + np.random.normal(size = num_data) * 0.1
chem2 = base_data**2 + np.random.normal(size = num_data) * 0.1
chem3 = 1 - base_data + np.random.normal(size = num_data) * 0.1
df_chem = pd.DataFrame({'chem1':chem1,'chem2':chem2,'chem3':chem3})

In [ ]:
sns.pairplot(df_chem)

In [ ]:
df_chem.corr()

### Answer

**Three graphs:** chem1 vs chem2, chem1 vs chem3, and chem2 vs chem3, the three unique scatter plots below the diagonal. The three above the diagonal are mirror copies of them, and the histograms only describe one variable each.

**Making them clearer:** label the axes with units (e.g. "Chemical A (mg/L)"), give each plot a title that states the finding (e.g. "A and C move in opposite directions"), use smaller, semi-transparent points so overlaps are visible, and print R on each panel.

**Fitting lines/curves:**
- **A vs C:** yes, a straight line. The relationship is clearly linear and negative (R ≈ -0.95).
- **A vs B:** a curve, not a line. B follows A², so a quadratic fit describes it better; a straight line would understate the relationship.
- **B vs C:** also a (decreasing) quadratic curve.

Only fit a model when you expect the relationship to follow that form; otherwise a fitted line can suggest more certainty than the data supports.

**Just one:** chem1 vs chem3, the strongest and simplest relationship and the easiest to explain.

**Six:** the three unique scatter plots plus the three histograms, so the audience also sees each chemical's distribution (e.g. chem2 is skewed toward low values). I'd still leave out the mirrored duplicates.

The cell below shows the three chosen plots with these improvements.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
pairs = [('chem1', 'chem2', 2), ('chem1', 'chem3', 1), ('chem2', 'chem3', 2)]
xs = np.linspace(df_chem.min().min(), df_chem.max().max(), 200)
for ax, (a, b, deg) in zip(axes, pairs):
    ax.scatter(df_chem[a], df_chem[b], alpha=0.6, s=20)
    coefs = np.polyfit(df_chem[a], df_chem[b], deg)
    xr = np.linspace(df_chem[a].min(), df_chem[a].max(), 200)
    ax.plot(xr, np.polyval(coefs, xr), color='crimson', linewidth=2,
            label='quadratic fit' if deg == 2 else 'linear fit')
    r = df_chem[a].corr(df_chem[b])
    ax.set_title(f'{a} vs {b}  (R = {r:.2f})')
    ax.set_xlabel(f'{a} level')
    ax.set_ylabel(f'{b} level')
    ax.legend()
plt.tight_layout()
plt.show()

# 3. Homework - Reproducing Pair Plot Graphs
Example: this cloud involves a match between two normal distributions.

In [ ]:
num_points = 1000
np.random.seed(0)
data1 = np.random.normal(size = num_points) + 2
data2 = np.random.normal(size = num_points) + 2
df_twonorm = pd.DataFrame({"data1":data1,"data2":data2})
df_twonorm.plot.scatter(x = "data1", y = "data2")

**Problem:** design data that will generate two vertically separated clouds of points.

In [ ]:
# Reference image (only displays if twoclouds.png is in the same folder as this notebook)
if os.path.exists("twoclouds.png"):
    display(Image("twoclouds.png"))

In [ ]:
# np.concatenate joins two series into one long series for the y-axis
np.random.seed(0)
n = 500
x = np.random.normal(loc=0, scale=1, size=2*n)
y = np.concatenate([np.random.normal(loc=0, scale=1, size=n),    # lower cloud
                    np.random.normal(loc=8, scale=1, size=n)])   # upper cloud
df_twoclouds = pd.DataFrame({"data1": x, "data2": y})
df_twoclouds.plot.scatter(x="data1", y="data2", alpha=0.5)

**Challenge problem!** Three segments of the first series connect with three segments of the second series.

In [ ]:
if os.path.exists("threeclouds.png"):
    display(Image("threeclouds.png"))

In [ ]:
# Adjust the loc values below to match the positions of the clouds in threeclouds.png
np.random.seed(0)
n = 300
data_A = np.concatenate([np.random.normal(0, 1, n),     # left cloud
                         np.random.normal(5, 1, n),     # middle cloud
                         np.random.normal(10, 1, n)])   # right cloud
data_B = np.concatenate([np.random.normal(0, 1, n),     # low
                         np.random.normal(10, 1, n),    # high
                         np.random.normal(0, 1, n)])    # low
df_three = pd.DataFrame({"data_A": data_A, "data_B": data_B})
df_three.plot.scatter(x="data_A", y="data_B", alpha=0.5)

Would you expect to see a correlation for this challenge problem? Test it with `corr()`.

In [ ]:
df_three.corr()

**Answer:** No. data_B is low at both small and large values of data_A and high only in the middle, so knowing whether data_A is large or small doesn't tell you whether data_B is large or small. R comes out close to 0.

The variables are still clearly related; the relationship just isn't *linear*, and correlation only measures linear relationships. This is why it's important to plot the data, not just compute R.

Here is a histogram. Provide data that approximately re-creates it: tailing off on either side, but tailing off to a nonzero number (about 300).

In [ ]:
if os.path.exists("histogram.png"):
    display(Image("histogram.png"))

In [ ]:
# A normal distribution creates the central peak; a uniform distribution adds a flat
# "floor" of about 3000 / 10 bins = 300 points per bin, so the tails never reach zero.
np.random.seed(0)
hist_data = np.concatenate([np.random.normal(loc=50, scale=10, size=6000),
                            np.random.uniform(0, 100, size=3000)])
pd.Series(hist_data).plot.hist(bins=10, edgecolor='white')

Create a line plot that shows a sales volume of soft drinks that decreases, then increases over the course of a year, in a U-shaped curve, but with some noise.

In [ ]:
np.random.seed(0)
months = pd.date_range(start='2023-01-01', periods=12, freq='MS')
t = np.arange(12)
sales = 50 + 2 * (t - 5.5)**2 + np.random.normal(scale=5, size=12)
ax = sns.lineplot(x=months, y=sales, marker='o')
ax.set(title="Soft drink sales volume, 2023", xlabel="Month", ylabel="Units sold (thousands)")
plt.xticks(rotation=45)
plt.show()

Create an area plot that shows the sales volume for two brands of soft drinks, one of which is increasing then decreasing, the other decreasing then increasing.

In [ ]:
np.random.seed(1)
brand_a = 100 - 2 * (t - 5.5)**2 + np.random.normal(scale=4, size=12)  # rises, then falls
brand_b = 40 + 2 * (t - 5.5)**2 + np.random.normal(scale=4, size=12)   # falls, then rises
df_brands = pd.DataFrame({"Brand A": brand_a, "Brand B": brand_b}, index=months)
ax = df_brands.plot.area(title="Soft drink sales by brand, 2023", alpha=0.7)
ax.set(xlabel="Month", ylabel="Units sold (thousands)")
plt.show()

# 4. Storytelling With Data plot
Reproducing the decluttered "Ticket volume over time" line chart from Chapter 3 ("Clutter is your enemy!"). The monthly values are approximate, so check them against your copy of the book.

In [ ]:
months_lbl = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
received  = [160,184,241,149,180,161,132,202,160,139,149,177]
processed = [160,184,237,148,181,150,123,156,126,104,124,140]

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(months_lbl, received,  color='#7f7f7f', linewidth=3)
ax.plot(months_lbl, processed, color='#1f77b4', linewidth=3)
ax.scatter([11, 11], [received[-1], processed[-1]], color=['#7f7f7f', '#1f77b4'], zorder=3)

# Declutter: remove chart border and gridlines, grey axes, label lines directly (no legend)
for side in ['top', 'right']:
    ax.spines[side].set_visible(False)
for side in ['left', 'bottom']:
    ax.spines[side].set_color('#bfbfbf')
ax.tick_params(colors='#7f7f7f')
ax.set_ylim(0, 300)
ax.set_xlim(-0.5, 12.5)

ax.text(11.3, received[-1],  f'{received[-1]} Received',  color='#7f7f7f', va='center', fontsize=11)
ax.text(11.3, processed[-1], f'{processed[-1]} Processed', color='#1f77b4', va='center', fontsize=11)

ax.set_title('Ticket volume over time', loc='left', fontsize=16, color='#404040', pad=15)
ax.set_ylabel('Number of tickets', color='#7f7f7f', loc='top')
ax.set_xlabel('2014', color='#7f7f7f', loc='left')
plt.tight_layout()
plt.show()